# Summary All Models From Drive

Notebook ini menjalankan bagian summary/visualisasi yang setara dengan:

```bash
python code_final_run_v2.py --phase summary --models all
```

Bedanya, notebook ini langsung membaca `all_results.csv` yang sudah ada di Drive dan tidak menjalankan preprocessing, training, atau evaluasi model.


In [ ]:

from __future__ import annotations

import importlib.util
import json
import os
from pathlib import Path
import sys
from datetime import datetime

import numpy as np
import pandas as pd

PIPELINE_STAGE_NAME = "code_v4_musicnet_cqtdiffplus_44k"
COLAB_DRIVE_ROOT = Path(os.environ.get("COLAB_DRIVE_ROOT", "/content/drive/MyDrive"))
DEFAULT_DRIVE_DATA_ROOT = COLAB_DRIVE_ROOT / "THESIS CODE"

TARGET_SR = int(os.environ.get("PIPELINE_TARGET_SR", "44100"))
SEGMENT_SAMPLES = int(os.environ.get("PIPELINE_SEGMENT_SAMPLES", "184184"))
SEGMENT_DURATION = SEGMENT_SAMPLES / TARGET_SR
EXPERIMENT_CONFIG_ID = (
    f"musicnet_cqtdiffplus_sr{TARGET_SR}_n{SEGMENT_SAMPLES}_"
    f"dur{SEGMENT_DURATION:.6f}s"
)
DATASET_FRACTION = float(os.environ.get("DATASET_FRACTION", "1.0"))
EVAL_GAP_POSITION = os.environ.get("EVAL_GAP_POSITION", "center").strip().lower()

EXPECTED_MODEL_CONFIGS = [
    "baseline_cqtdiff",
    "baseline_cqtdiff_finetuned",
    "clap_cqtdiff",
    "clap_maid",
    "audiomae_cqtdiff",
    "audiomae_maid",
]


In [ ]:
def info(message: str) -> None:
    print(f"[summary-all] {message}", flush=True)


def fail(message: str) -> None:
    raise SystemExit(f"\nERROR: {message}\n")


def is_colab_runtime() -> bool:
    return (
        "COLAB_GPU" in os.environ
        or "google.colab" in sys.modules
        or importlib.util.find_spec("google.colab") is not None
    )


def maybe_mount_google_drive() -> None:
    if not is_colab_runtime():
        return
    if COLAB_DRIVE_ROOT.exists():
        info(f"Google Drive already mounted: {COLAB_DRIVE_ROOT}")
        return
    try:
        from google.colab import drive
    except Exception as exc:
        fail(f"Runtime Colab terdeteksi, tapi Drive tidak bisa dimount. Detail: {exc}")
    info("Mounting Google Drive at /content/drive")
    drive.mount("/content/drive")


In [ ]:

def project_root() -> Path:
    if "PROJECT_ROOT" in os.environ:
        return Path(os.environ["PROJECT_ROOT"]).resolve()
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    return Path.cwd().resolve()


def data_root() -> Path:
    if "MUSIC_INPAINTING_ROOT" in os.environ:
        return Path(os.environ["MUSIC_INPAINTING_ROOT"]).resolve()
    if is_colab_runtime():
        return DEFAULT_DRIVE_DATA_ROOT.resolve()
    return (project_root() / "music_inpainting").resolve()


def path_config() -> dict[str, Path]:
    all_results_override = os.environ.get("ALL_RESULTS_CSV", "").strip()
    results_dir_override = os.environ.get("SUMMARY_RESULTS_DIR", "").strip()

    if all_results_override:
        all_results_csv = Path(all_results_override).expanduser().resolve()
        results_dir = all_results_csv.parent
        stage_root = results_dir.parent
        base = stage_root.parent.parent if stage_root.parent.name == "training_stages" else data_root()
    elif results_dir_override:
        results_dir = Path(results_dir_override).expanduser().resolve()
        all_results_csv = results_dir / "all_results.csv"
        stage_root = results_dir.parent
        base = stage_root.parent.parent if stage_root.parent.name == "training_stages" else data_root()
    else:
        base = data_root()
        stage_root = base / "training_stages" / PIPELINE_STAGE_NAME
        results_dir = stage_root / "results"
        all_results_csv = results_dir / "all_results.csv"

    return {
        "base": base,
        "stage": stage_root,
        "results": results_dir,
        "plots": stage_root / "plots",
        "logs": stage_root / "logs",
        "all_results_csv": all_results_csv,
    }


In [ ]:

def evaluation_artifact_name(model_name: str) -> str:
    if EVAL_GAP_POSITION == "center":
        return model_name
    return f"{model_name}_{EVAL_GAP_POSITION}gap"


def format_duration(seconds):
    seconds = float(seconds or 0.0)
    hours, rem = divmod(int(seconds), 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours:d}h {minutes:02d}m {secs:02d}s"
    if minutes:
        return f"{minutes:d}m {secs:02d}s"
    return f"{seconds:.1f}s"


def safe_float(value, default=None):
    try:
        if value is None or pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def make_json_safe(value):
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if value is pd.NA:
        return None
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def _read_single_timing_seconds(path: Path, column: str):
    if path.exists():
        try:
            df = pd.read_csv(path)
            if not df.empty and column in df.columns:
                return safe_float(df[column].iloc[-1], 0.0)
        except Exception:
            return 0.0
    return 0.0


In [ ]:

def update_experiment_summary(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    timing_path = paths["results"] / "training_timing_summary.csv"
    eval_timing_path = paths["results"] / "evaluation_timing_summary.csv"
    results_path = paths["all_results_csv"]
    preprocessing_path = paths["results"] / "preprocessing_timing.csv"

    timing_df = pd.read_csv(timing_path) if timing_path.exists() else pd.DataFrame()
    eval_df = pd.read_csv(eval_timing_path) if eval_timing_path.exists() else pd.DataFrame()
    results_df = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()
    preprocessing_seconds = _read_single_timing_seconds(preprocessing_path, "preprocessing_seconds")

    for model_name in EXPECTED_MODEL_CONFIGS:
        row = {
            "stage": PIPELINE_STAGE_NAME,
            "model": model_name,
            "dataset_fraction": DATASET_FRACTION,
            "experiment_config_id": EXPERIMENT_CONFIG_ID,
            "target_sr": TARGET_SR,
            "segment_samples": SEGMENT_SAMPLES,
            "segment_duration": SEGMENT_DURATION,
            "eval_gap_position": EVAL_GAP_POSITION,
            "batch_size": None,
            "epochs": None,
            "training_seconds": None,
            "training_time": None,
            "preprocessing_seconds": preprocessing_seconds,
            "evaluation_seconds": None,
            "evaluation_time": None,
            "peak_vram_gb": None,
            "checkpoint_path": None,
            "final_LSD_mean": None,
            "final_LSD_GAP_ONLY_mean": None,
            "final_GAP_LSD_mean": None,
            "final_GAP_SI_SDR_mean": None,
            "final_GAP_SNR_mean": None,
            "final_GAP_MEL_DISTANCE_mean": None,
            "final_FAD_mean": None,
            "final_VISQOL_ODG_mean": None,
            "final_PEAQ_ODG_mean": None,
            "status": "pending",
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

        if not timing_df.empty and "model" in timing_df.columns:
            m = timing_df[timing_df["model"] == model_name]
            if not m.empty:
                last = m.iloc[-1]
                row.update({
                    "batch_size": last.get("batch_size"),
                    "epochs": last.get("num_epochs"),
                    "training_seconds": safe_float(last.get("total_seconds")),
                    "training_time": last.get("total_time"),
                    "peak_vram_gb": safe_float(last.get("peak_vram_gb")),
                    "checkpoint_path": last.get("checkpoint_path"),
                    "status": last.get("status", "trained"),
                })

        if not eval_df.empty and "model" in eval_df.columns:
            e = eval_df[eval_df["model"] == model_name]
            if "eval_gap_position" in e.columns:
                e = e[e["eval_gap_position"].fillna("center") == EVAL_GAP_POSITION]
            if not e.empty:
                last = e.iloc[-1]
                row["evaluation_seconds"] = safe_float(last.get("evaluation_seconds"))
                row["evaluation_time"] = last.get("evaluation_time")
                row["status"] = "evaluated"

        if not results_df.empty and "model" in results_df.columns:
            r = results_df[results_df["model"] == model_name]
            if "gap_position" in r.columns:
                r = r[r["gap_position"].fillna("center") == EVAL_GAP_POSITION]
            if not r.empty:
                for metric in [
                    "LSD", "LSD_GAP_ONLY", "GAP_LSD", "GAP_SI_SDR", "GAP_SNR",
                    "GAP_MEL_DISTANCE", "FAD", "VISQOL_ODG", "PEAQ_ODG",
                    "GAP_WINDOW_VISQOL_ODG",
                ]:
                    if metric in r.columns:
                        row[f"final_{metric}_mean"] = safe_float(r[metric].mean())
                row["status"] = "evaluated"

        rows.append(row)

    summary_df = pd.DataFrame(rows)
    paths["results"].mkdir(parents=True, exist_ok=True)
    csv_path = paths["results"] / "experiment_summary.csv"
    json_path = paths["results"] / "experiment_summary.json"
    summary_df.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(make_json_safe(rows), indent=2), encoding="utf-8")
    print(f"Experiment summary saved: {csv_path}")
    print(f"Experiment summary JSON saved: {json_path}")
    return summary_df


In [ ]:
def run_summary(paths: dict[str, Path]) -> pd.DataFrame | None:
    import matplotlib
    import matplotlib.pyplot as plt

    matplotlib.rcParams["figure.dpi"] = 150
    matplotlib.rcParams["font.size"] = 10

    master_path = paths["all_results_csv"]
    if not master_path.exists():
        print("? File hasil belum ada. Pastikan evaluasi model sudah dijalankan.")
        print(f"Expected: {master_path}")
        return None

    all_results = pd.read_csv(master_path)

    if "VISQOL_ODG" not in all_results.columns:
        if "VISQOL" in all_results.columns:
            all_results["VISQOL_ODG"] = all_results["VISQOL"]
    if "gap_position" not in all_results.columns:
        all_results["gap_position"] = "center"
    all_results = all_results[all_results["gap_position"].fillna("center") == EVAL_GAP_POSITION].copy()
    if all_results.empty:
        print(f"? Tidak ada hasil untuk gap_position={EVAL_GAP_POSITION}.")
        return None

    available_models = all_results["model"].unique()
    print(f"?? Available models: {list(available_models)}")

    gap_durations = sorted(all_results["gap_ms"].unique())

    print("\n" + "="*70)
    print("FULL COMPARISON TABLE")
    print("="*70)

    metric_order = [
        m for m in [
            "LSD", "LSD_GAP_ONLY", "GAP_SI_SDR", "GAP_SNR", "GAP_MEL_DISTANCE",
            "FAD", "VISQOL_ODG", "PEAQ_ODG", "GAP_WINDOW_VISQOL_ODG",
        ]
        if m in all_results.columns
    ]
    for metric in metric_order:
        if metric in ["LSD", "LSD_GAP_ONLY", "GAP_LSD", "GAP_MEL_DISTANCE", "FAD"]:
            direction = "lower is better"
        elif metric in ["GAP_SI_SDR", "GAP_SNR"]:
            direction = "higher is better"
        else:
            direction = "closer to 0 is better"
        print(f"\n{metric} ({direction}):")
        pivot = all_results.pivot(index="gap_ms", columns="model", values=metric)
        ordered_cols = [c for c in [
            "baseline_cqtdiff", "baseline_cqtdiff_finetuned",
            "clap_cqtdiff", "audiomae_cqtdiff", "clap_maid", "audiomae_maid",
        ] if c in pivot.columns]
        print(pivot[ordered_cols].to_string())

    styles = {
        "baseline_cqtdiff":  {"color": "#000000", "marker": "x", "linestyle": "--",
                               "label": "Baseline: CQT-Diff+ pretrained", "linewidth": 2.5, "zorder": 10},
        "baseline_cqtdiff_finetuned": {"color": "#666666", "marker": "P", "linestyle": "-.",
                               "label": "Baseline: CQT-Diff+ fine-tuned no SSL", "linewidth": 2.2, "zorder": 9},
        "clap_cqtdiff":      {"color": "#2196F3", "marker": "o", "linestyle": "-",
                               "label": "CLAP + CQT-Diff+", "linewidth": 1.5, "zorder": 5},
        "clap_maid":         {"color": "#4CAF50", "marker": "s", "linestyle": "-",
                               "label": "CLAP + MAID", "linewidth": 1.5, "zorder": 5},
        "audiomae_cqtdiff":  {"color": "#FF9800", "marker": "^", "linestyle": "-",
                               "label": "AudioMAE + CQT-Diff+", "linewidth": 1.5, "zorder": 5},
        "audiomae_maid":     {"color": "#F44336", "marker": "D", "linestyle": "-",
                               "label": "AudioMAE + MAID", "linewidth": 1.5, "zorder": 5},
    }

    metrics_info = {
        "LSD": {"title": "Log Spectral Distance (LSD)",
                "ylabel": "LSD (dB)",
                "note": "lower is better"},
        "LSD_GAP_ONLY": {"title": "Gap-only Log Spectral Distance",
                "ylabel": "Gap LSD (dB)",
                "note": "lower is better"},
        "GAP_SI_SDR": {"title": "Gap SI-SDR",
                "ylabel": "SI-SDR (dB)",
                "note": "higher is better"},
        "GAP_SNR": {"title": "Gap SNR",
                "ylabel": "SNR (dB)",
                "note": "higher is better"},
        "GAP_MEL_DISTANCE": {"title": "Gap Mel Spectral Distance",
                "ylabel": "Mean |Mel dB diff|",
                "note": "lower is better"},
        "FAD": {"title": "Frechet Audio Distance (FAD)",
                "ylabel": "FAD Score",
                "note": "lower is better"},
        "VISQOL_ODG": {"title": "ViSQOL Objective Difference Grade",
                       "ylabel": "VISQOL_ODG Score",
                       "note": "closer to 0 is better"},
        "PEAQ_ODG": {"title": "GstPEAQ Objective Difference Grade",
                     "ylabel": "PEAQ_ODG Score",
                     "note": "closer to 0 is better"},
        "GAP_WINDOW_VISQOL_ODG": {"title": "Gap-window ViSQOL ODG",
                     "ylabel": "Gap-window VISQOL_ODG",
                     "note": "closer to 0 is better"},
    }
    metrics_info = {k: v for k, v in metrics_info.items() if k in metric_order}

    if metrics_info:
        fig, axes = plt.subplots(1, len(metrics_info), figsize=(6 * len(metrics_info), 6))
        if len(metrics_info) == 1:
            axes = [axes]
        fig.suptitle(
            "Music Audio Inpainting ? Baseline vs Hybrid SSL+Diffusion Models\n"
            f"Native MusicNet CQTdiff+: {TARGET_SR} Hz, {SEGMENT_SAMPLES} samples ({SEGMENT_DURATION:.2f}s), "
            f"gap={EVAL_GAP_POSITION}",
            fontsize=13, fontweight="bold"
        )

        for ax, (metric, info_dict) in zip(axes, metrics_info.items()):
            plot_order = [m for m in [
                "clap_cqtdiff", "audiomae_cqtdiff", "clap_maid", "audiomae_maid",
                "baseline_cqtdiff_finetuned", "baseline_cqtdiff",
            ] if m in available_models]

            for model_name in plot_order:
                model_data = all_results[all_results["model"] == model_name].sort_values("gap_ms")
                s = styles.get(model_name, {"color": "gray", "marker": "x",
                                             "linestyle": "-", "label": model_name,
                                             "linewidth": 1.5, "zorder": 1})
                ax.plot(
                    model_data["gap_ms"],
                    model_data[metric],
                    color=s["color"],
                    marker=s["marker"],
                    linestyle=s["linestyle"],
                    label=s["label"],
                    linewidth=s["linewidth"],
                    markersize=7,
                    zorder=s["zorder"],
                )

            ax.set_title(info_dict["title"], fontsize=11, fontweight="bold")
            ax.set_xlabel("Gap Duration (ms)", fontsize=10)
            ax.set_ylabel(info_dict["ylabel"], fontsize=10)
            ax.set_xticks(gap_durations)
            ax.set_xticklabels([str(g) for g in gap_durations], rotation=45)
            ax.legend(fontsize=8, loc="best")
            ax.grid(True, alpha=0.3)
            ax.text(0.02, 0.98, info_dict["note"],
                    transform=ax.transAxes, fontsize=8,
                    verticalalignment="top", style="italic", color="gray")

        plt.tight_layout()
        paths["plots"].mkdir(parents=True, exist_ok=True)
        plot_path = paths["plots"] / "comparison_plot.png"
        plt.savefig(plot_path, bbox_inches="tight", dpi=150)
        plt.show()
        print(f"\n?? Grafik disimpan: {plot_path}")

    if "baseline_cqtdiff" in available_models:
        print("\n" + "="*72)
        print("?? IMPROVEMENT HYBRID vs PRETRAINED BASELINE (per gap duration)")
        print("   Positive = better than baseline")
        print("="*72)

        baseline_data = all_results[all_results["model"] == "baseline_cqtdiff"]
        for gap_ms in gap_durations:
            bl = baseline_data[baseline_data["gap_ms"] == gap_ms].iloc[0]
            print(f"\n  Gap {gap_ms}ms:")
            header_cols = ["Model", "?LSD"]
            if "FAD" in all_results.columns:
                header_cols.append("?FAD")
            if "VISQOL_ODG" in all_results.columns:
                header_cols.append("?VISQOL_ODG")
            if "PEAQ_ODG" in all_results.columns:
                header_cols.append("?PEAQ_ODG")
            print(f"  {header_cols[0]:<25} " + " ".join(f"{h:>10}" for h in header_cols[1:]))
            print(f"  {'-'*65}")

            for model_name in ["clap_cqtdiff", "clap_maid", "audiomae_cqtdiff", "audiomae_maid"]:
                if model_name not in available_models:
                    continue
                hybrid = all_results[
                    (all_results["model"] == model_name) &
                    (all_results["gap_ms"] == gap_ms)
                ].iloc[0]
                deltas = [bl["LSD"] - hybrid["LSD"]]
                if "FAD" in all_results.columns:
                    deltas.append(bl["FAD"] - hybrid["FAD"])
                if "VISQOL_ODG" in all_results.columns:
                    deltas.append(hybrid["VISQOL_ODG"] - bl["VISQOL_ODG"])
                if "PEAQ_ODG" in all_results.columns:
                    deltas.append(hybrid["PEAQ_ODG"] - bl["PEAQ_ODG"])
                print(f"  {model_name:<25} " + " ".join(f"{d:>+10.4f}" for d in deltas))

    if "baseline_cqtdiff_finetuned" in available_models:
        print("\n" + "="*78)
        print("?? CQT HYBRID vs FINE-TUNED NO-SSL BASELINE (fair SSL contribution check)")
        print("   Positive = hybrid is better than the fine-tuned no-SSL baseline")
        print("="*78)

        ft_data = all_results[all_results["model"] == "baseline_cqtdiff_finetuned"]
        fair_metrics = [
            m for m in ["LSD_GAP_ONLY", "GAP_SI_SDR", "GAP_SNR", "GAP_MEL_DISTANCE", "FAD", "PEAQ_ODG"]
            if m in all_results.columns
        ]
        for gap_ms in gap_durations:
            ft = ft_data[ft_data["gap_ms"] == gap_ms].iloc[0]
            print(f"\n  Gap {gap_ms}ms:")
            print(f"  {'Model':<25} " + " ".join(f"?{m:>14}" for m in fair_metrics))
            print(f"  {'-'*90}")
            for model_name in ["clap_cqtdiff", "audiomae_cqtdiff"]:
                if model_name not in available_models:
                    continue
                hybrid = all_results[
                    (all_results["model"] == model_name) &
                    (all_results["gap_ms"] == gap_ms)
                ].iloc[0]
                deltas = []
                for metric in fair_metrics:
                    if metric in ["LSD", "LSD_GAP_ONLY", "GAP_LSD", "GAP_MEL_DISTANCE", "FAD"]:
                        deltas.append(ft[metric] - hybrid[metric])
                    else:
                        deltas.append(hybrid[metric] - ft[metric])
                print(f"  {model_name:<25} " + " ".join(f"{d:>+15.4f}" for d in deltas))

    summary_df = update_experiment_summary(paths)
    print(f"\n? Semua hasil tersimpan di: {paths['results']}")
    print(f"Plots tersimpan di: {paths['plots']}")
    return summary_df


In [ ]:

def main() -> pd.DataFrame | None:
    maybe_mount_google_drive()
    paths = path_config()
    info(f"base root      : {paths['base']}")
    info(f"stage root     : {paths['stage']}")
    info(f"all_results    : {paths['all_results_csv']}")
    info(f"results folder : {paths['results']}")
    info(f"plots folder   : {paths['plots']}")
    return run_summary(paths)


In [ ]:

summary_df = main()
summary_df
